# Reddit Mental Health Data — JSONL to CSV

**Course:** MGS 3001 WHS01  
**Author:** Kexin Huang  
**Date:** May 2026

---

### What This Notebook Does

This notebook loads raw Reddit post data downloaded from the **Arctic Shift archive** (https://arctic-shift.photon-reddit.com/download-tool) in JSONL format, selects the variables needed for analysis, performs basic preprocessing, and exports a clean CSV file.

### The Pipeline

```
JSONL file → Load → Select variables → Clean text → Compute derived fields → Export CSV
```

### Why Arctic Shift instead of the Reddit API?

Since June 2023, Reddit has imposed strict rate limits and access restrictions on third-party API usage. The Reddit official API no longer supports bulk historical data collection at the scale required for this study. Arctic Shift provides complete subreddit archives in JSONL format with consistent field coverage, and is widely used in recent academic research on Reddit data (e.g., del Rio-Chanona et al., 2024; Shan & Qiu, 2025).

---

## Cell 1: Import Libraries

In [16]:
import json
import pandas as pd
from pathlib import Path
from datetime import datetime

print("Libraries imported successfully!")

Libraries imported successfully!


## Cell 2: Define File Paths and Key Dates

We define the two cutoff dates used in our Regression Discontinuity Design:
- **Cutoff 1**: November 30, 2022 — ChatGPT public release
- **Cutoff 2**: September 30, 2024 — Widespread ChatGPT adoption (Chatterji et al., 2025)

In [17]:
# 时间配置
CUTOFF_DATE1 = '2022-11-30'  # ChatGPT发布日
CUTOFF_DATE2 = '2024-09-30' # 通过文献调研出来，chatgpt普遍使用的日期 # How People Use ChatGPT，应该使用这之后的数据 2024-9-30
START_DATE = '2021-09-01'
END_DATE = '2026-04-26'

CHUNK_SIZE=10000

# File path — update this to match your local file location
JSONL_FILE = Path('./data/r_mentalhealth_posts.jsonl')

# Output CSV path
OUTPUT_CSV = Path('./data/r_mentalhealth_posts_clean.csv')

# Key dates for RDD analysis
CUTOFF_DATE1 = pd.Timestamp(CUTOFF_DATE1)  # ChatGPT release
CUTOFF_DATE2 = pd.Timestamp(CUTOFF_DATE2)  # Widespread adoption

print(f"Input file : {JSONL_FILE}")
print(f"Output file: {OUTPUT_CSV}")
print(f"Cutoff 1   : {CUTOFF_DATE1.date()} (ChatGPT release)")
print(f"Cutoff 2   : {CUTOFF_DATE2.date()} (Widespread adoption)")

Input file : data\r_mentalhealth_posts.jsonl
Output file: data\r_mentalhealth_posts_clean.csv
Cutoff 1   : 2022-11-30 (ChatGPT release)
Cutoff 2   : 2024-09-30 (Widespread adoption)


## Cell 3: Inspect the Raw JSONL File

Each line in the JSONL file is one Reddit post stored as a JSON object.
We first look at the first record to understand the available fields.

The raw data contains **108 fields** — we only need a small subset for our analysis.

In [18]:
# Read and display the first record
with open(JSONL_FILE, 'r', encoding='utf-8') as f:
    first_record = json.loads(f.readline())

print(f"Total fields in one record: {len(first_record)}")
print(f"\nAll available field names:")
for key in sorted(first_record.keys()):
    print(f"  {key}")


Total fields in one record: 91

All available field names:
  all_awardings
  allow_live_comments
  archived
  author
  author_created_utc
  author_flair_background_color
  author_flair_css_class
  author_flair_richtext
  author_flair_template_id
  author_flair_text
  author_flair_text_color
  author_flair_type
  author_fullname
  author_patreon_flair
  author_premium
  awarders
  banned_by
  can_gild
  can_mod_post
  category
  content_categories
  contest_mode
  created_utc
  discussion_type
  distinguished
  domain
  edited
  gilded
  gildings
  hidden
  hide_score
  id
  is_created_from_ads_ui
  is_crosspostable
  is_meta
  is_original_content
  is_reddit_media_domain
  is_robot_indexable
  is_self
  is_video
  link_flair_background_color
  link_flair_css_class
  link_flair_richtext
  link_flair_template_id
  link_flair_text
  link_flair_text_color
  link_flair_type
  locked
  media
  media_embed
  media_only
  name
  no_follow
  num_comments
  num_crossposts
  over_18
  parent_whit

In [19]:
# Preview the fields we actually need
FIELDS_NEEDED = ['id', 'subreddit', 'title', 'selftext', 'created_utc',
                 'score', 'ups', 'num_comments']

print("Fields we will extract and their values from the first record:\n")
for field in FIELDS_NEEDED:
    value = first_record.get(field, 'NOT FOUND')
    # Truncate long strings for display
    if isinstance(value, str) and len(value) > 80:
        value = value[:80] + '...'
    print(f"  {field:20s}: {value}")

Fields we will extract and their values from the first record:

  id                  : pfi62h
  subreddit           : mentalhealth
  title               : At what point do I quit?
  selftext            : I am absolutely miserable at work. I’ve been with the same company for 15 years,...
  created_utc         : 1630454772
  score               : 3
  ups                 : 3
  num_comments        : 3


## Cell 4: Load the Full JSONL File

We load all records, keeping only the 8 raw fields we need.
This avoids loading 108 columns into memory unnecessarily.

In [20]:
# FIELDS_NEEDED = ['id', 'subreddit', 'title', 'selftext', 'created_utc',
#                  'score', 'ups', 'num_comments']
#
# records = []
#
# with open(JSONL_FILE, 'r', encoding='utf-8') as f:
#     for line in f:
#         line = line.strip()
#         if not line:
#             continue
#         record = json.loads(line)
#         # Select only the fields we need
#         selected = {field: record.get(field, None) for field in FIELDS_NEEDED}
#         records.append(selected)
# df = pd.DataFrame(records)


def load_jsonl(filepath: Path) -> pd.DataFrame:
    """加载JSONL文件（分块处理大文件）"""
    data = []
    # 先尝试读取前几行判断是否需要分块
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            for i, line in enumerate(f):
                if i >= 100:  # 如果超过100行，使用分块读取
                    return load_jsonl_chunked(filepath)
                if line.strip():
                    data.append(json.loads(line))
        return pd.DataFrame(data)
    except:
        return load_jsonl_chunked(filepath)

def load_jsonl_chunked(filepath: Path) -> pd.DataFrame:
    """分块加载大文件, 当然也可以 with open(filepath, 'w') as f:
    for line in f:
    if line:
    .......
    """
    chunks = []
    for chunk in pd.read_json(filepath, lines=True, chunksize=CHUNK_SIZE):
        chunks.append(chunk)
    return pd.concat(chunks, ignore_index=True)

df = load_jsonl(JSONL_FILE) # posts or comments
df = pd.DataFrame(df)  # 重建 DataFrame
df = df.copy(deep=True)
print(f"Loaded {len(df):,} records")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst 3 rows:")
df.head(3)

Loaded 565,354 records
Columns: ['all_awardings', 'allow_live_comments', 'archived', 'author', 'author_created_utc', 'author_flair_background_color', 'author_flair_css_class', 'author_flair_richtext', 'author_flair_template_id', 'author_flair_text', 'author_flair_text_color', 'author_flair_type', 'author_fullname', 'author_patreon_flair', 'author_premium', 'awarders', 'banned_by', 'can_gild', 'can_mod_post', 'category', 'content_categories', 'contest_mode', 'created_utc', 'discussion_type', 'distinguished', 'domain', 'edited', 'gilded', 'gildings', 'hidden', 'hide_score', 'id', 'is_created_from_ads_ui', 'is_crosspostable', 'is_meta', 'is_original_content', 'is_reddit_media_domain', 'is_robot_indexable', 'is_self', 'is_video', 'link_flair_background_color', 'link_flair_css_class', 'link_flair_richtext', 'link_flair_template_id', 'link_flair_text', 'link_flair_text_color', 'link_flair_type', 'locked', 'media', 'media_embed', 'media_only', 'name', 'no_follow', 'num_comments', 'num_crosspo

,all_awardings,allow_live_comments,archived,author,author_created_utc,author_flair_background_color,author_flair_css_class,author_flair_richtext,author_flair_template_id,author_flair_text,...,user_reports,visited,_meta,previous_selftext,selftext_html,location_lat,location_long,location_name,websocket_url,outbound_link
0,[],False,False,CertainlyAmbivalent,1.600638e+09,None,NaN,[],None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,[],False,False,[deleted],NaN,,NaN,NaN,None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,[],False,False,[deleted],NaN,,NaN,NaN,None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Cell 5: Clean and Preprocess

Three cleaning steps:
1. **Convert timestamp** — `created_utc` is stored as Unix epoch (integer seconds), convert to datetime
2. **Combine text** — merge `title` and `selftext` into a single `text` field for analysis
3. **Filter deleted posts** — Reddit marks removed content as `[removed]` or `[deleted]`; these contain no useful text

In [21]:
# Step 1: Convert Unix timestamp to datetime
df['date'] = pd.to_datetime(df['created_utc'], unit='s', utc=True)
df['date'] = df['date'].dt.tz_localize(None)  # remove timezone info for simplicity

print("Date range in dataset:")
print(f"  Earliest: {df['date'].min()}")
print(f"  Latest  : {df['date'].max()}")

Date range in dataset:
  Earliest: 2021-09-01 00:06:12
  Latest  : 2026-04-28 09:39:10


In [22]:
# Step 2: Combine title + selftext into a single 'text' field
# This matches the _combine_text() method in the main analysis code
df['selftext'] = df['selftext'].fillna('')
df['title'] = df['title'].fillna('')
df['text'] = (df['title'] + ' ' + df['selftext']).str.strip()

print(f"Sample combined text from first record:")
print(f"  {df['text'].iloc[0][:200]}...")

Sample combined text from first record:
  At what point do I quit? I am absolutely miserable at work. I’ve been with the same company for 15 years, mostly decent. But recently they’ve made changes which have greatly impacted my team in negati...


In [23]:
# Step 3: Filter out deleted/removed posts and posts with very short text
MIN_TEXT_LENGTH = 20  # consistent with Config in analysis code

before = len(df)

# Remove deleted/removed content
df = df[~df['text'].str.contains(r'\[removed\]|\[deleted\]', regex=True, na=False)]

# Remove posts with very short text (likely low-quality or image-only)
df = df[df['text'].str.len() >= MIN_TEXT_LENGTH]

after = len(df)
print(f"Rows before filtering : {before:,}")
print(f"Rows after filtering  : {after:,}")
print(f"Removed               : {before - after:,} ({(before-after)/before*100:.1f}%)")

Rows before filtering : 565,354
Rows after filtering  : 431,303
Removed               : 134,051 (23.7%)


## Cell 6: Add Derived Fields

We add fields that the main analysis code computes:
- `type` — marks these as posts (vs comments, which come from a separate file)
- `period` — 'before' or 'after' the ChatGPT cutoff
- `days_from_cutoff1` — running variable for RDD (negative = before ChatGPT, positive = after)
- `days_from_cutoff2` — running variable for secondary cutoff (widespread adoption)

In [24]:
# Mark content type
df['type'] = 'post'

# Period label relative to ChatGPT release
df['period'] = df['date'].apply(lambda d: 'after' if d >= CUTOFF_DATE1 else 'before')

# Days from cutoff (the RDD running variable)
df['days_from_cutoff1'] = (df['date'] - CUTOFF_DATE1).dt.days
df['days_from_cutoff2'] = (df['date'] - CUTOFF_DATE2).dt.days

print("Period distribution:")
print(df['period'].value_counts())
print(f"\ndays_from_cutoff1 range: {df['days_from_cutoff1'].min()} to {df['days_from_cutoff1'].max()}")
print(f"\ndays_from_cutoff1 range: {df['days_from_cutoff2'].min()} to {df['days_from_cutoff2'].max()}")


Period distribution:
period
after     347442
before     83861
Name: count, dtype: int64

days_from_cutoff1 range: -455 to 1245

days_from_cutoff1 range: -1125 to 575


## Cell 7: Final Column Selection and Preview

We keep the columns in the order the main analysis code expects them.
The three feature columns (`psychoedu_score`, `psych_term_density`, `support_score`) are **not included here** — they will be computed by the `FeatureExtractor` class in the main analysis pipeline.

In [25]:
# Final column order (matches USE_COLS in main analysis code)
FINAL_COLS = [
    'id',                  # Unique post identifier
    'subreddit',           # Source subreddit
    'type',                # 'post' (vs 'comment' from separate file)
    'date',                # Datetime (converted from created_utc)
    'text',                # Combined title + selftext
    'score',               # Reddit score (upvotes - downvotes)
    'ups',                 # Raw upvote count
    'num_comments',        # Number of comments on the post
    'period',              # 'before' or 'after' ChatGPT release
    'days_from_cutoff1',   # RDD running variable (Cutoff 1: ChatGPT release)
    'days_from_cutoff2',   # RDD running variable (Cutoff 2: widespread adoption)
]

df_final = df[FINAL_COLS].copy()

print(f"Final dataset shape: {df_final.shape[0]:,} rows × {df_final.shape[1]} columns")
print(f"\nColumn summary:")
df_final.info()

Final dataset shape: 431,303 rows × 11 columns

Column summary:
<class 'pandas.core.frame.DataFrame'>
Index: 431303 entries, 0 to 565353
Data columns (total 11 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   id                 431303 non-null  object        
 1   subreddit          431303 non-null  object        
 2   type               431303 non-null  object        
 3   date               431303 non-null  datetime64[ns]
 4   text               431303 non-null  object        
 5   score              431303 non-null  int64         
 6   ups                431303 non-null  int64         
 7   num_comments       431303 non-null  int64         
 8   period             431303 non-null  object        
 9   days_from_cutoff1  431303 non-null  int64         
 10  days_from_cutoff2  431303 non-null  int64         
dtypes: datetime64[ns](1), int64(5), object(5)
memory usage: 39.5+ MB


In [26]:
# Preview first 5 rows
df_final.head()

,id,subreddit,type,date,text,score,ups,num_comments,period,days_from_cutoff1,days_from_cutoff2
0,pfi62h,mentalhealth,post,2021-09-01 00:06:12,At what point do I quit? I am absolutely miser...,3,3,3,before,-455,-1125
2,pficjx,mentalhealth,post,2021-09-01 00:16:22,"High functioning depression? Hey guys, been di...",13,13,6,before,-455,-1125
4,pfidml,mentalhealth,post,2021-09-01 00:18:14,Does anyone else have their brain blank out af...,5,5,1,before,-455,-1125
5,pfie4i,mentalhealth,post,2021-09-01 00:19:02,"I want to sleep for a very long time I know, o...",12,12,8,before,-455,-1125
7,pfiirc,mentalhealth,post,2021-09-01 00:27:13,Can too much of one color make you anxious? Th...,2,2,1,before,-455,-1125


## Cell 8: Descriptive Statistics

Basic statistics for the key numeric variables — this is what goes into the Assignment 3 report.

In [27]:
# Descriptive statistics for numeric variables
numeric_cols = ['score', 'ups', 'num_comments', 'days_from_cutoff1']
print("Descriptive Statistics (numeric variables):")
df_final[numeric_cols].describe().round(2)

Descriptive Statistics (numeric variables):


,score,ups,num_comments,days_from_cutoff1
count,431303.00,431303.00,431303.00,431303.00
mean,3.88,3.88,3.15,527.73
std,21.54,21.54,11.09,480.42
min,0.00,0.00,0.00,-455.00
25%,1.00,1.00,0.00,166.00
50%,1.00,1.00,1.00,582.00
75%,2.00,2.00,3.00,941.00
max,2344.00,2344.00,1040.00,1245.00


In [28]:
# Text length distribution
df_final['text_length'] = df_final['text'].str.len()
print("Text length (characters):")
print(df_final['text_length'].describe().round(0))

print(f"\nPosts by period:")
print(df_final['period'].value_counts())

print(f"\nDate range:")
print(f"  Earliest: {df_final['date'].min().date()}")
print(f"  Latest  : {df_final['date'].max().date()}")

Text length (characters):
count    431303.0
mean       1132.0
std        1063.0
min          20.0
25%         458.0
50%         855.0
75%        1483.0
max       40094.0
Name: text_length, dtype: float64

Posts by period:
period
after     347442
before     83861
Name: count, dtype: int64

Date range:
  Earliest: 2021-09-01
  Latest  : 2026-04-28


## Cell 9: Export to CSV

In [29]:
# Create output directory if it doesn't exist
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

# Export (without the temporary text_length column)
df_final[FINAL_COLS].to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
# Save 1000 rows for uploading to github
df_final[FINAL_COLS][:1000].to_csv(OUTPUT_CSV.parent / 'r_mentalhealth_posts_clean_1000.csv', index=False, encoding='utf-8')
print(f"✅ Saved to: {OUTPUT_CSV}")
print(f"   Rows   : {len(df_final):,}")
print(f"   Columns: {len(FINAL_COLS)}")
print(f"   File size: {OUTPUT_CSV.stat().st_size / 1024:.1f} KB")

✅ Saved to: data\r_mentalhealth_posts_clean.csv
   Rows   : 431,303
   Columns: 11
   File size: 510794.0 KB



## Summary

| Step | Description | Result |
|------|-------------|--------|
| Load | Read JSONL, select 8 raw fields | ~1,000 rows (this file is a sample) |
| Clean | Convert timestamp, combine text, filter deleted | Removed [removed]/[deleted] posts |
| Derive | Add type, period, days_from_cutoff1/2 | 11 final columns |
| Export | Save to CSV | `r_mentalhealth_posts_clean.csv` |

**Next step:** Run the same notebook for the other three subreddits (`r/anxiety`, `r/depression`, `r/psychology`) and for comments files, then concatenate all into `combined_df.csv` as the main analysis input.
